# Dataset · Data

Saved snapshot · DuckDB SQL + Pandas tables · Run All. [DuckDB notebooks](https://duckdb.org/docs/current/guides/python/jupyter).

In [ ]:
import json
import sys
import unicodedata
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/plazapd/mappedin.py").is_file()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or nbs/plazapd.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.plazapd.mappedin import load_tables

DATA_DIR = ROOT / "data/plazapd"
tables = load_tables(DATA_DIR, format="parquet")
manifest = json.loads((DATA_DIR / "manifest.json").read_text())


def text_key(value):
    return "".join(c for c in unicodedata.normalize("NFKD", value).casefold() if c.isalnum())


venue_aliases = {text_key(value) for value in [
    manifest.get("venue_name"), manifest.get("venue_id"), *tables["venues"].name.dropna()
] if isinstance(value, str) and value}


def venue_label(value):
    if isinstance(value, str) and any(alias in text_key(value) for alias in venue_aliases):
        return "Venue"
    return value


for table in tables.values():
    for column in ("name", "description"):
        if column in table:
            table[column] = table[column].map(venue_label).astype(table[column].dtype)

assert tables["entities"].entity_id.is_unique
assert tables["relationships"][["src", "dst"]].isin(set(tables["entities"].entity_id)).all().all()
assert tables["distances"].loc[tables["distances"].status.eq("unreachable"), "distance_m"].isna().all()
pd.options.display.max_colwidth = 50
pd.options.display.max_rows = 25

db = duckdb.connect(":memory:")
for name, frame in tables.items():
    db.register(name, frame)


def sql(query, parameters=None):
    return db.execute(query, parameters).df()

## Inventory

Stores and amenities are subsets of locations; locations and structural collections are subsets of entities.

In [ ]:
display(pd.Series({k: manifest.get(k) for k in ["collected_at", "map_updated_at"]}, name="snapshot").to_frame())
inventory = pd.DataFrame([{"table": name, "rows": len(t), "columns": len(t.columns)} for name, t in tables.items()])
display(inventory)
display(sql("SELECT collection, type, count(*) AS entities FROM entities GROUP BY ALL ORDER BY entities DESC"))
display(sql("SELECT relation, count(*) AS edges FROM relationships GROUP BY ALL ORDER BY edges DESC"))
display(sql("SELECT entity_id, name FROM venues"))

In [ ]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW location_spaces AS
SELECT DISTINCT r.src AS location_id, r.dst AS space_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN spaces s ON s.entity_id = r.dst WHERE r.relation = 'spaces';

CREATE OR REPLACE TEMP VIEW space_floors AS
SELECT DISTINCT r.src AS space_id, r.dst AS floor_entity_id
FROM relationships r JOIN spaces s ON s.entity_id = r.src
JOIN floors f ON f.entity_id = r.dst WHERE r.relation = 'floor';

CREATE OR REPLACE TEMP VIEW location_floors AS
SELECT DISTINCT l.location_id, f.floor_entity_id
FROM location_spaces l JOIN space_floors f USING (space_id);

CREATE OR REPLACE TEMP VIEW location_category_links AS
SELECT DISTINCT r.src AS location_id, r.dst AS category_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN categories c ON c.entity_id = r.dst WHERE r.relation = 'categories';

CREATE OR REPLACE TEMP VIEW node_catalog AS
SELECT e.entity_id, e.id, coalesce(nullif(trim(e.name), ''), e.entity_id) AS label,
       CASE WHEN s.entity_id IS NOT NULL THEN 'store'
            WHEN a.entity_id IS NOT NULL THEN 'amenity'
            WHEN e.collection = 'spaces' THEN 'space'
            WHEN e.collection = 'floors' THEN 'floor'
            WHEN e.collection = 'categories' THEN 'category'
            WHEN e.collection = 'nodes' THEN 'navigation'
            ELSE e.type END AS kind,
       e.hidden, e.navigable, e.amenity
FROM entities e LEFT JOIN stores s USING (entity_id) LEFT JOIN amenities a USING (entity_id);
""")

## Stores and amenities × floor

Distinct locations per floor; multi-floor locations contribute to each linked floor.

In [ ]:
floor_counts = sql("""
SELECT f.entity_id, f.name AS floor, f.elevation,
       count(DISTINCT l.location_id) FILTER (WHERE n.kind = 'store') AS stores,
       count(DISTINCT l.location_id) FILTER (WHERE n.kind = 'amenity') AS amenities
FROM floors f LEFT JOIN location_floors l ON l.floor_entity_id = f.entity_id
LEFT JOIN node_catalog n ON n.entity_id = l.location_id
GROUP BY ALL ORDER BY f.elevation, f.entity_id
""")
display(floor_counts)

## Categories × floor

Counts use distinct stores; multi-category and multi-floor memberships overlap.

In [ ]:
category_floor = sql("""
SELECT c.name AS category, f.name AS floor, count(DISTINCT s.entity_id) AS stores
FROM stores s JOIN location_category_links lc ON lc.location_id = s.entity_id
JOIN categories c ON c.entity_id = lc.category_id
JOIN location_floors lf ON lf.location_id = s.entity_id
JOIN floors f ON f.entity_id = lf.floor_entity_id
GROUP BY ALL ORDER BY category, floor
""")
floor_order = tables["floors"].sort_values(["elevation", "entity_id"]).name.tolist()
display(category_floor.pivot(index="category", columns="floor", values="stores").reindex(
    index=sorted(tables["categories"].name), columns=floor_order).fillna(0).astype(int))
display(sql("""
SELECT c.name AS category, count(DISTINCT s.entity_id) AS stores
FROM categories c LEFT JOIN location_category_links lc ON lc.category_id = c.entity_id
LEFT JOIN stores s ON s.entity_id = lc.location_id GROUP BY ALL ORDER BY stores DESC, category
"""))

## Stores with multiple spaces or floors

In [ ]:
store_spatial = sql("""
SELECT s.entity_id, s.name, count(DISTINCT ls.space_id) AS spaces,
       count(DISTINCT lf.floor_entity_id) AS floors,
       string_agg(DISTINCT f.name, ' | ' ORDER BY f.name) AS floor_names
FROM stores s LEFT JOIN location_spaces ls ON ls.location_id = s.entity_id
LEFT JOIN location_floors lf ON lf.location_id = s.entity_id
LEFT JOIN floors f ON f.entity_id = lf.floor_entity_id
GROUP BY ALL ORDER BY floors DESC, spaces DESC, s.name, s.entity_id
""")
display(store_spatial.loc[store_spatial.spaces.gt(1) | store_spatial.floors.gt(1)])
display(store_spatial.loc[store_spatial.spaces.eq(0), ["entity_id", "name", "spaces", "floors"]])

## Missing links and navigation coverage

In [ ]:
link_coverage = sql("""
SELECT n.kind, count(*) AS locations,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_spaces x WHERE x.location_id=n.entity_id)) AS no_space,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_floors x WHERE x.location_id=n.entity_id)) AS no_floor,
 count(*) FILTER (WHERE NOT EXISTS (SELECT 1 FROM location_category_links x WHERE x.location_id=n.entity_id)) AS no_category,
 count(*) FILTER (WHERE n.hidden) AS hidden,
 count(*) FILTER (WHERE n.navigable) AS navigable,
 count(*) FILTER (WHERE n.navigable IS NULL) AS navigation_unknown
FROM node_catalog n WHERE kind IN ('store','amenity') GROUP BY kind ORDER BY kind
""")
display(link_coverage)
display(sql("SELECT hidden, navigable, count(*) AS stores FROM stores GROUP BY ALL ORDER BY ALL"))

## Directed walking distances · metres

Missing route rows and explicit unreachable pairs are counted separately; unreachable distances remain null.

In [ ]:
routing_coverage = sql("""
SELECT count(*) AS navigable_locations, count(*) * (count(*) - 1) AS expected_pairs,
       (SELECT count(*) FROM distances) AS stored_pairs,
       count(*) * (count(*) - 1) - (SELECT count(*) FROM distances) AS missing_pairs
FROM locations WHERE navigable
""")
display(routing_coverage)
display(sql("SELECT status, count(*) AS pairs, count(distance_m) AS measured FROM distances GROUP BY status ORDER BY status"))
display(sql("""
SELECT min(distance_m) AS min_m, quantile_cont(distance_m,0.25) AS p25_m,
 quantile_cont(distance_m,0.5) AS median_m, quantile_cont(distance_m,0.75) AS p75_m,
 quantile_cont(distance_m,0.9) AS p90_m, max(distance_m) AS max_m
FROM distances WHERE status='reachable'
""").round(2))

## Nearest reachable amenity per store

Coverage denominators are navigable stores; zero metres is a measured result.

In [ ]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW nearest_amenity AS
SELECT s.entity_id, s.name, s.navigable, a.entity_id AS amenity_id,
       a.name AS amenity_name, a.amenity AS amenity_type, d.distance_m
FROM stores s LEFT JOIN distances d ON d.from_id=s.id AND d.status='reachable'
 AND d.to_id IN (SELECT id FROM amenities)
LEFT JOIN amenities a ON a.id=d.to_id
QUALIFY row_number() OVER (PARTITION BY s.entity_id ORDER BY d.distance_m NULLS LAST, a.entity_id)=1
""")
amenity_coverage = sql("""
SELECT count(*) AS stores, count(*) FILTER (WHERE navigable) AS eligible_stores,
 count(distance_m) AS with_reachable_amenity,
 count(*) FILTER (WHERE navigable AND distance_m IS NULL) AS eligible_without_reachable_amenity,
 quantile_cont(distance_m,0.5) AS median_nearest_m, max(distance_m) AS max_nearest_m
FROM nearest_amenity
""")
display(amenity_coverage.round(2))
nearest_distance_summary = sql("""
SELECT min(distance_m) AS min_m, quantile_cont(distance_m,0.25) AS p25_m,
 quantile_cont(distance_m,0.5) AS median_m, quantile_cont(distance_m,0.75) AS p75_m,
 quantile_cont(distance_m,0.9) AS p90_m, max(distance_m) AS max_m
FROM nearest_amenity WHERE distance_m IS NOT NULL
""")
display(nearest_distance_summary.round(2))
display(sql("SELECT name, amenity_name, amenity_type, round(distance_m,2) AS distance_m FROM nearest_amenity WHERE distance_m IS NOT NULL ORDER BY distance_m DESC, entity_id LIMIT 10"))
display(sql("""
SELECT n.name, n.navigable,
 CASE WHEN NOT coalesce(n.navigable,false) THEN 'Not a routing target'
      WHEN EXISTS (SELECT 1 FROM distances d JOIN stores s ON s.id=d.from_id WHERE s.entity_id=n.entity_id)
      THEN 'Route rows exist; no reachable amenity' ELSE 'No route rows' END AS route_coverage
FROM nearest_amenity n WHERE distance_m IS NULL ORDER BY n.name, n.entity_id
"""))

## Nearest amenities × type

Each type uses all navigable stores as its denominator.

In [ ]:
amenity_type_coverage = sql("""
WITH types AS (SELECT DISTINCT coalesce(nullif(amenity,''),'Unspecified') AS amenity_type FROM amenities),
eligibility AS (SELECT count(*) AS eligible_stores FROM stores WHERE navigable),
nearest AS (
 SELECT d.from_id, coalesce(nullif(a.amenity,''),'Unspecified') AS amenity_type, min(d.distance_m) AS nearest_m
 FROM distances d JOIN stores s ON s.id=d.from_id AND s.navigable
 JOIN amenities a ON a.id=d.to_id WHERE d.status='reachable' GROUP BY ALL
)
SELECT t.amenity_type, e.eligible_stores,
 count(n.from_id) AS with_reachable_amenity, eligible_stores-count(n.from_id) AS without_reachable_amenity,
 round(100.0*count(n.from_id)/nullif(eligible_stores,0),1) AS coverage_pct,
 round(quantile_cont(n.nearest_m,0.5),2) AS median_nearest_m, round(max(n.nearest_m),2) AS max_nearest_m
FROM types t CROSS JOIN eligibility e LEFT JOIN nearest n USING (amenity_type)
GROUP BY t.amenity_type, e.eligible_stores ORDER BY coverage_pct, t.amenity_type
""")
display(amenity_type_coverage)

## Forward versus reverse routes

Each unordered pair appears once; distance differences use a 0.01-metre tolerance.

In [ ]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW route_comparison AS
SELECT a.from_id, a.to_id, a.status AS forward_status, b.status AS reverse_status,
       a.distance_m AS forward_m, b.distance_m AS reverse_m,
       abs(a.distance_m-b.distance_m) AS difference_m
FROM distances a JOIN distances b ON a.from_id=b.to_id AND a.to_id=b.from_id
WHERE a.from_id<a.to_id
""")
route_asymmetry = sql("""
SELECT count(*) AS unordered_pairs,
 count(*) FILTER (WHERE forward_status<>reverse_status) AS unequal_reachability,
 count(*) FILTER (WHERE forward_status='reachable' AND reverse_status='reachable') AS both_reachable,
 count(*) FILTER (WHERE difference_m>0.01) AS distance_diff_over_1cm
FROM route_comparison
""")
display(route_asymmetry)
display(sql("""
SELECT f.name AS source, t.name AS destination, round(r.forward_m,2) AS forward_m,
 round(r.reverse_m,2) AS reverse_m, round(r.difference_m,2) AS difference_m
FROM route_comparison r JOIN locations f ON f.id=r.from_id JOIN locations t ON t.id=r.to_id
WHERE r.difference_m>0.01 ORDER BY r.difference_m DESC, r.from_id, r.to_id LIMIT 10
"""))